In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 🍊 BINARY CITRUS LEAF CLASSIFIER - PyTorch + MobileNetV3
# Dataset: 120k images (60k citrus_leaf + 60k not_citrus_leaf)
# ══════════════════════════════════════════════════════════════════════════════

# ── Cellule 1 : Vérification GPU ──────────────────────────────────────────────
import torch
import subprocess

print('=' * 60)
print('🔍  VÉRIFICATION GPU')
print('=' * 60)

cuda_ok = torch.cuda.is_available()
print(f'  CUDA disponible         : {cuda_ok}')

if cuda_ok:
    print(f'  Nom du GPU              : {torch.cuda.get_device_name(0)}')
    print(f'  Mémoire totale          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
    print(f'  CUDA version            : {torch.version.cuda}')
    print(f'  cuDNN version           : {torch.backends.cudnn.version()}')
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    print('  cuDNN benchmark         : ACTIVÉ ✅')
else:
    print('  ⚠️  Aucun GPU détecté — entraînement sur CPU (très lent)')

print('=' * 60)


🔍  VÉRIFICATION GPU
  CUDA disponible         : True
  Nom du GPU              : Tesla T4
  Mémoire totale          : 15.64 GB
  CUDA version            : 12.8
  cuDNN version           : 91002
  cuDNN benchmark         : ACTIVÉ ✅


In [ ]:
# ── Cellule 2 : Montage Google Drive ──────────────────────────────────────────

from google.colab import drive
import time

print('=' * 60)
print('📂  MONTAGE GOOGLE DRIVE')
print('=' * 60)

t0 = time.time()
drive.mount('/content/drive')
print(f'  ✅ Drive monté en {time.time()-t0:.1f}s')
print('=' * 60)

📂  MONTAGE GOOGLE DRIVE
Mounted at /content/drive
  ✅ Drive monté en 14.9s


In [ ]:
# ── Cellule 3 : Extraction dataset depuis ZIP (I/O rapide) ──────────────────
import zipfile, os, time

print('=' * 60)
print('📦  EXTRACTION DU DATASET DEPUIS ZIP')
print('=' * 60)

zip_src = '/content/drive/MyDrive/citrus_binary_dataset.zip'
dst     = '/content/dataset'

if os.path.exists(dst):
    print(f'  Dossier {dst} déjà présent — extraction ignorée.')
else:
    print(f'  Source  : {zip_src}')
    print(f'  Dest    : {dst}')
    print('  Extraction en cours...')
    t0 = time.time()
    with zipfile.ZipFile(zip_src, 'r') as zf:
        zf.extractall('/content')
    print(f'  ✅ Extraction terminée en {time.time()-t0:.1f}s')

# Compter les images par split
for split in ['train', 'val', 'test']:
    split_path = os.path.join(dst, split)
    if os.path.exists(split_path):
        classes = [c for c in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, c))]
        n_imgs  = sum(len(os.listdir(os.path.join(split_path, c))) for c in classes)
        print(f'  {split:5s} → {len(classes)} classes | {n_imgs} images')
        for c in sorted(classes):
            n = len(os.listdir(os.path.join(split_path, c)))
            print(f'           • {c:<20} : {n} images')
    else:
        print(f'  ⚠️  {split_path} n\'existe pas!')

print('=' * 60)

📦  EXTRACTION DU DATASET DEPUIS ZIP
  Source  : /content/drive/MyDrive/citrus_binary_dataset.zip
  Dest    : /content/dataset
  Extraction en cours...
  ✅ Extraction terminée en 72.4s
  ⚠️  /content/dataset/train n'existe pas!
  ⚠️  /content/dataset/val n'existe pas!
  ⚠️  /content/dataset/test n'existe pas!


In [ ]:
# Compter les images par split
for split in ['train', 'val', 'test']:
    split_path = os.path.join(dst, split)
    if os.path.exists(split_path):
        classes = [c for c in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, c))]
        n_imgs  = sum(len(os.listdir(os.path.join(split_path, c))) for c in classes)
        print(f'  {split:5s} → {len(classes)} classes | {n_imgs} images')
        for c in sorted(classes):
            n = len(os.listdir(os.path.join(split_path, c)))
            print(f'           • {c:<20} : {n} images')
    else:
        print(f'  ⚠️  {split_path} n\'existe pas!')

print('=' * 60)

  train → 2 classes | 84000 images
           • citrus_leaf          : 42000 images
           • not_citrus_leaf      : 42000 images
  val   → 2 classes | 18000 images
           • citrus_leaf          : 9000 images
           • not_citrus_leaf      : 9000 images
  test  → 2 classes | 18000 images
           • citrus_leaf          : 9000 images
           • not_citrus_leaf      : 9000 images


In [ ]:
# ── Cellule 4 : Configuration des chemins ─────────────────────────────────────
import os

print('=' * 60)
print('📁  CONFIGURATION DES CHEMINS')
print('=' * 60)

# ✅ CORRIGÉ : pointe vers le dataset local (copié depuis Drive)
# Structure attendue:
# /content/dataset/
#   ├── train/
#   │   ├── citrus_leaf/
#   │   └── not_citrus_leaf/
#   ├── val/
#   │   ├── citrus_leaf/
#   │   └── not_citrus_leaf/
#   └── test/
#       ├── citrus_leaf/
#       └── not_citrus_leaf/

DATA_DIR = '/content/dataset'  # ✅ Local SSD — beaucoup plus rapide que Drive

# Vérification
for split in ['train', 'val', 'test']:
    split_path = os.path.join(DATA_DIR, split)
    if os.path.exists(split_path):
        classes = [c for c in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, c))]
        n_imgs = sum(len(os.listdir(os.path.join(split_path, c))) for c in classes)
        print(f'  {split:5s} → {len(classes)} classes | {n_imgs:,} images')
    else:
        print(f'  ⚠️  {split_path} n\'existe pas!')

print('=' * 60)

📁  CONFIGURATION DES CHEMINS
  train → 2 classes | 84,000 images
  val   → 2 classes | 18,000 images
  test  → 2 classes | 18,000 images


In [ ]:
# ── Cellule 5 : Data pipeline avec augmentation ───────────────────────────────
from torchvision import datasets, transforms
from torch.utils.data import DataLoaderkk
import multiprocessing

print('=' * 60)
print('⚙️   CONFIGURATION DATA PIPELINE')
print('=' * 60)

NUM_WORKERS = min(multiprocessing.cpu_count(), 8)
BATCH_SIZE = 64
PIN_MEMORY = True
PREFETCH = 4

print(f'  Batch size              : {BATCH_SIZE}')
print(f'  Num workers             : {NUM_WORKERS}')
print(f'  Pin memory              : {PIN_MEMORY}')
print(f'  Prefetch factor         : {PREFETCH}')

# Augmentation renforcée pour train
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.RandomGrayscale(p=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])


val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_transforms)
val_dataset   = datasets.ImageFolder(f'{DATA_DIR}/val',   transform=val_transforms)
test_dataset  = datasets.ImageFolder(f'{DATA_DIR}/test',  transform=val_transforms)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    prefetch_factor=PREFETCH, persistent_workers=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    prefetch_factor=PREFETCH, persistent_workers=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    prefetch_factor=PREFETCH, persistent_workers=False
)

print(f'  Classes ({len(train_dataset.classes)})            : {train_dataset.classes}')
print(f'  Train images            : {len(train_dataset):,}')
print(f'  Val images              : {len(val_dataset):,}')
print(f'  Test images             : {len(test_dataset):,}')
print(f'  Batches train           : {len(train_loader)}')
print(f'  Batches val             : {len(val_loader)}')
print('  ✅ DataLoaders prêts')
print('=' * 60)

⚙️   CONFIGURATION DATA PIPELINE
  Batch size              : 64
  Num workers             : 2
  Pin memory              : True
  Prefetch factor         : 4
  Classes (2)            : ['citrus_leaf', 'not_citrus_leaf']
  Train images            : 84,000
  Val images              : 18,000
  Test images             : 18,000
  Batches train           : 1313
  Batches val             : 282
  ✅ DataLoaders prêts


In [ ]:
# ── Cellule 6 : Modèle MobileNetV3 pour classification binaire ────────────────
import torchvision.models as models
import torch.nn as nn

print('=' * 60)
print('🧠  CHARGEMENT MODÈLE MOBILENETV3-LARGE (Binary)')
print('=' * 60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'  Device                  : {device}')

# install mobileNet3
model = models.mobilenet_v3_large(weights='IMAGENET1K_V1')

# Gel de toutes les couches
for param in model.parameters():
    param.requires_grad = False

# Dégel des 3 derniers blocs de features + classifier
for param in model.features[13:].parameters():
    param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True

# ⚠️ IMPORTANT : Classification binaire = 1 seule sortie avec sigmoid
# On remplace la dernière couche (1000 classes) par 1 sortie
in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, 1)  # Binary: 1 output

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

model = model.to(device)

print(f'  Paramètres totaux       : {total_params:,}')
print(f'  Paramètres entraînables : {trainable_params:,}')
print(f'  Sortie                  : 1 neurone (sigmoid pour binaire)')
print(f'  Couches dégelées        : features[13:] + classifier')

if torch.cuda.is_available():
    mem_alloc = torch.cuda.memory_allocated(0) / 1e6
    print(f'  VRAM après chargement   : {mem_alloc:.1f} MB')

print('  ✅ Modèle chargé sur', device)
print('=' * 60)

🧠  CHARGEMENT MODÈLE MOBILENETV3-LARGE (Binary)
  Device                  : cuda
  Paramètres totaux       : 4,203,313
  Paramètres entraînables : 3,410,825
  Sortie                  : 1 neurone (sigmoid pour binaire)
  Couches dégelées        : features[13:] + classifier
  VRAM après chargement   : 165.6 MB
  ✅ Modèle chargé sur cuda


In [ ]:
# ── Cellule 7 : Fonctions train/eval avec AMP ─────────────────────────────────
from torch.cuda.amp import autocast, GradScaler
import time

scaler = GradScaler()

def gpu_stats():
    if not torch.cuda.is_available():
        return ''
    alloc    = torch.cuda.memory_allocated(0) / 1e6
    reserved = torch.cuda.memory_reserved(0) / 1e6
    return f'VRAM {alloc:.0f}/{reserved:.0f} MB'


def train_one_epoch(model, loader, optimizer, criterion, epoch, total_epochs):
    model.train()
    total_loss = 0.0
    correct    = 0
    total      = 0
    t_start    = time.time()

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float().unsqueeze(1)  # [B, 1]

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            outputs = model(images)  # [B, 1]
            loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds       = (torch.sigmoid(outputs) > 0.5).float()  # Binary predictions
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(loader):
            elapsed    = time.time() - t_start
            imgs_per_s = (batch_idx + 1) * images.size(0) / elapsed
            print(
                f'    Epoch {epoch}/{total_epochs}'
                f' | Batch {batch_idx+1:3d}/{len(loader)}'
                f' | Loss {total_loss/(batch_idx+1):.4f}'
                f' | Train Acc {correct/total:.3f}'
                f' | {imgs_per_s:.0f} img/s'
                f' | {gpu_stats()}'
            )

    epoch_time = time.time() - t_start
    return total_loss / len(loader), correct / total, epoch_time


def evaluate(model, loader):
    model.eval()
    correct = 0
    total   = 0
    t_start = time.time()

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).float().unsqueeze(1)

            with autocast():
                outputs = model(images)

            preds    = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    val_time = time.time() - t_start
    return correct / total, val_time

print('  ✅ Fonctions train/eval définies avec AMP (float16)')


  ✅ Fonctions train/eval définies avec AMP (float16)


/tmp/ipykernel_2275/3128422448.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [24]:
# ── Cellule 8 : Entraînement ──────────────────────────────────────────────────
import torch.optim as optim

EPOCHS         = 5
LR_BACKBONE    = 1e-4
LR_CLASSIFIER  = 3e-4

backbone_params   = list(model.features[13:].parameters())
classifier_params = list(model.classifier.parameters())

optimizer = optim.Adam([
    {'params': backbone_params,   'lr': LR_BACKBONE},
    {'params': classifier_params, 'lr': LR_CLASSIFIER},
])

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# ⚠️ Pour classification binaire: BCEWithLogitsLoss (combine sigmoid + BCE)
criterion = nn.BCEWithLogitsLoss()

history  = []
best_acc = 0.0
total_t  = time.time()

print('=' * 60)
print('🏋️   DÉMARRAGE ENTRAÎNEMENT (Binary Classification)')
print('=' * 60)
print(f'  Epochs                  : {EPOCHS}')
print(f'  LR backbone (features)  : {LR_BACKBONE}')
print(f'  LR classifier           : {LR_CLASSIFIER}')
print(f'  Scheduler               : CosineAnnealingLR (eta_min=1e-6)')
print(f'  Loss                    : BCEWithLogitsLoss (binary)')
print(f'  Mixed Precision (AMP)   : ✅ activé')
print(f'  Fine-tuning             : features[13:] + classifier dégelés')
print('=' * 60)

for epoch in range(1, EPOCHS + 1):
    lr_bb  = optimizer.param_groups[0]['lr']
    lr_cls = optimizer.param_groups[1]['lr']
    print(f'\n📌 Epoch {epoch}/{EPOCHS}  —  LR backbone={lr_bb:.2e} | LR classifier={lr_cls:.2e}')
    print(f'  ▶ Phase TRAIN ...')

    train_loss, train_acc, t_train = train_one_epoch(
        model, train_loader, optimizer, criterion, epoch, EPOCHS
    )

    print(f'  ▶ Phase VAL ...')
    val_acc, t_val = evaluate(model, val_loader)

    scheduler.step()

    # Sauvegarde du meilleur modèle
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_citrus_binary_model.pth')
        best_tag = '  ⭐ NOUVEAU MEILLEUR MODÈLE SAUVEGARDÉ'
    else:
        best_tag = ''

    history.append({'epoch': epoch, 'loss': train_loss,
                    'train_acc': train_acc, 'val_acc': val_acc})

    if torch.cuda.is_available():
        vram_peak = torch.cuda.max_memory_allocated(0) / 1e6
        torch.cuda.reset_peak_memory_stats(0)
        vram_str = f'  | VRAM peak {vram_peak:.0f} MB'
    else:
        vram_str = ''

    print(
        f'\n  ✅ Epoch {epoch:2d} résumé'
        f' | Loss {train_loss:.4f}'
        f' | Train Acc {train_acc:.4f}'
        f' | Val Acc {val_acc:.4f}'
        f' | Train {t_train:.1f}s'
        f' | Val {t_val:.1f}s'
        f'{vram_str}'
        f'{best_tag}'
    )

print('\n' + '=' * 60)
print(f'  🏁 Entraînement terminé en {time.time()-total_t:.1f}s')
print(f'  🏆 Meilleure Val Acc : {best_acc:.4f}')
print(f'  💾 Meilleur modèle  : best_citrus_binary_model.pth')
print('=' * 60)

🏋️   DÉMARRAGE ENTRAÎNEMENT (Binary Classification)
  Epochs                  : 5
  LR backbone (features)  : 0.0001
  LR classifier           : 0.0003
  Scheduler               : CosineAnnealingLR (eta_min=1e-6)
  Loss                    : BCEWithLogitsLoss (binary)
  Mixed Precision (AMP)   : ✅ activé
  Fine-tuning             : features[13:] + classifier dégelés

📌 Epoch 1/5  —  LR backbone=1.00e-04 | LR classifier=3.00e-04
  ▶ Phase TRAIN ...


/tmp/ipykernel_2275/3128422448.py:28: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


    Epoch 1/5 | Batch  10/1313 | Loss 0.4816 | Train Acc 0.795 | 63 img/s | VRAM 245/694 MB
    Epoch 1/5 | Batch  20/1313 | Loss 0.3406 | Train Acc 0.869 | 74 img/s | VRAM 245/694 MB
    Epoch 1/5 | Batch  30/1313 | Loss 0.2553 | Train Acc 0.905 | 75 img/s | VRAM 245/694 MB
    Epoch 1/5 | Batch  40/1313 | Loss 0.2133 | Train Acc 0.921 | 77 img/s | VRAM 245/694 MB
    Epoch 1/5 | Batch  50/1313 | Loss 0.1801 | Train Acc 0.933 | 77 img/s | VRAM 245/694 MB
    Epoch 1/5 | Batch  60/1313 | Loss 0.1589 | Train Acc 0.941 | 77 img/s | VRAM 245/694 MB
    Epoch 1/5 | Batch  70/1313 | Loss 0.1419 | Train Acc 0.947 | 79 img/s | VRAM 245/694 MB
    Epoch 1/5 | Batch  80/1313 | Loss 0.1298 | Train Acc 0.952 | 78 img/s | VRAM 245/694 MB
    Epoch 1/5 | Batch  90/1313 | Loss 0.1183 | Train Acc 0.956 | 79 img/s | VRAM 245/694 MB
    Epoch 1/5 | Batch 100/1313 | Loss 0.1084 | Train Acc 0.960 | 79 img/s | VRAM 245/694 MB
    Epoch 1/5 | Batch 110/1313 | Loss 0.1012 | Train Acc 0.963 | 80 img/s | VRAM

/tmp/ipykernel_2275/3128422448.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



  ✅ Epoch  1 résumé | Loss 0.0242 | Train Acc 0.9916 | Val Acc 0.9987 | Train 1091.2s | Val 44.1s  | VRAM peak 831 MB  ⭐ NOUVEAU MEILLEUR MODÈLE SAUVEGARDÉ

📌 Epoch 2/5  —  LR backbone=9.05e-05 | LR classifier=2.71e-04
  ▶ Phase TRAIN ...
    Epoch 2/5 | Batch  10/1313 | Loss 0.0089 | Train Acc 0.997 | 80 img/s | VRAM 245/700 MB
    Epoch 2/5 | Batch  20/1313 | Loss 0.0058 | Train Acc 0.998 | 75 img/s | VRAM 245/700 MB
    Epoch 2/5 | Batch  30/1313 | Loss 0.0046 | Train Acc 0.999 | 77 img/s | VRAM 245/700 MB
    Epoch 2/5 | Batch  40/1313 | Loss 0.0042 | Train Acc 0.999 | 75 img/s | VRAM 245/700 MB
    Epoch 2/5 | Batch  50/1313 | Loss 0.0048 | Train Acc 0.999 | 77 img/s | VRAM 245/700 MB
    Epoch 2/5 | Batch  60/1313 | Loss 0.0072 | Train Acc 0.998 | 76 img/s | VRAM 245/700 MB
    Epoch 2/5 | Batch  70/1313 | Loss 0.0067 | Train Acc 0.998 | 77 img/s | VRAM 245/700 MB
    Epoch 2/5 | Batch  80/1313 | Loss 0.0068 | Train Acc 0.998 | 76 img/s | VRAM 245/700 MB
    Epoch 2/5 | Batch  9

In [25]:
# ── Cellule 9 : Résumé final ──────────────────────────────────────────────────
print('=' * 60)
print('📊  RÉSUMÉ FINAL')
print('=' * 60)
print(f'  {"Epoch":>5}  {"Loss":>8}  {"Train Acc":>10}  {"Val Acc":>10}')
print('  ' + '-' * 40)
for h in history:
    best_mark = ' ⭐' if h['val_acc'] == best_acc else ''
    print(f'  {h["epoch"]:>5}  {h["loss"]:>8.4f}  {h["train_acc"]:>10.4f}  {h["val_acc"]:>10.4f}{best_mark}')
print('=' * 60)


📊  RÉSUMÉ FINAL
  Epoch      Loss   Train Acc     Val Acc
  ----------------------------------------
      1    0.0242      0.9916      0.9987
      2    0.0084      0.9971      0.9988
      3    0.0055      0.9981      0.9986
      4    0.0042      0.9986      0.9993
      5    0.0026      0.9991      0.9994 ⭐


In [26]:
# ── Cellule 10 : Évaluation sur test set ──────────────────────────────────────
print('\n' + '=' * 60)
print('🧪  ÉVALUATION SUR LE JEU DE TEST')
print('=' * 60)

# Charger le meilleur modèle
model.load_state_dict(torch.load('best_citrus_binary_model.pth'))
test_acc, test_time = evaluate(model, test_loader)

print(f'  Test Accuracy           : {test_acc:.4f}')
print(f'  Test Time               : {test_time:.1f}s')

# Matrice de confusion binaire
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

all_preds  = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device).float().unsqueeze(1)

        with autocast():
            outputs = model(images)

        preds = (torch.sigmoid(outputs) > 0.5).float()
        all_preds.extend(preds.cpu().numpy().flatten())
        all_labels.extend(labels.cpu().numpy().flatten())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print('\n📋  RAPPORT DE CLASSIFICATION')
print(classification_report(
    all_labels, all_preds,
    target_names=['not_citrus_leaf', 'citrus_leaf']
))

print('🔢  MATRICE DE CONFUSION')
cm = confusion_matrix(all_labels, all_preds)
print(f'                   Prédit: not_citrus  Prédit: citrus')
print(f'  Vrai: not_citrus        {cm[0,0]:>6}           {cm[0,1]:>6}')
print(f'  Vrai: citrus            {cm[1,0]:>6}           {cm[1,1]:>6}')

print('=' * 60)
print('  ✅ Entraînement et test terminés!')
print('=' * 60)



🧪  ÉVALUATION SUR LE JEU DE TEST


/tmp/ipykernel_2275/3128422448.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Test Accuracy           : 0.9992
  Test Time               : 45.2s


/tmp/ipykernel_2275/763145192.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



📋  RAPPORT DE CLASSIFICATION
                 precision    recall  f1-score   support

not_citrus_leaf       1.00      1.00      1.00      9000
    citrus_leaf       1.00      1.00      1.00      9000

       accuracy                           1.00     18000
      macro avg       1.00      1.00      1.00     18000
   weighted avg       1.00      1.00      1.00     18000

🔢  MATRICE DE CONFUSION
                   Prédit: not_citrus  Prédit: citrus
  Vrai: not_citrus          8995                5
  Vrai: citrus                 9             8991
  ✅ Entraînement et test terminés!


In [27]:
# ── Cellule 11 : Fonction de prédiction ───────────────────────────────────────
from PIL import Image

def predict_single_image(image_path, model, device, threshold=0.5):
    """
    Prédit si une image est une feuille d'agrume ou non

    Args:
        image_path: Chemin vers l'image
        model: Modèle entraîné
        device: 'cuda' ou 'cpu'
        threshold: Seuil de décision (default: 0.5)

    Returns:
        prediction: 'citrus_leaf' ou 'not_citrus_leaf'
        confidence: Score de confiance (0-1)
    """
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    img        = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        with autocast():
            output = model(img_tensor)
        prob = torch.sigmoid(output).item()

    if prob > threshold:
        prediction = 'citrus_leaf'
        confidence = prob
    else:
        prediction = 'not_citrus_leaf'
        confidence = 1 - prob

    print(f'Prédiction: {prediction}')
    print(f'Confiance: {confidence:.2%}')

    return prediction, confidence

# Exemple d'utilisation:
# prediction, confidence = predict_single_image('test_image.jpg', model, device)